In [1]:
from astropy.io import fits
import numpy as np
from astropy.table import Table
import pandas as pd

In [2]:
hdul = fits.open("CIV-Absorbers-dr1-v1.0.fits") # lya quasar catalog
catalog_absorbers = hdul[1].data
catalog_absorbers = catalog_absorbers['Z_ABS']
catalog_IDs = hdul[2].data['TARGETID'].astype(str)  # Ensure TARGETID is string type

df = pd.read_csv("CIV_catalog.csv", dtype={0:str})
detected_IDs = df.iloc[:,0]
detected_absorbers = df.iloc[:,[1,2,3,4,5,6,7]]

In [3]:
# print(catalog_IDs)
# print(detected_IDs)
# print(catalog_absorbers)
# print(detected_absorbers)
print(type(detected_IDs[1]))
print(detected_IDs[1])

<class 'str'>
39627791523648651


In [9]:
dv = 350 # km/s, velocity difference threshold
# Convert dv to redshift difference
c = 299792.458  # speed of light in km/s
z_trh = dv / c

TrueP = 0
TrueP2 = 0
FalseP = 0
FalseN = 0

# ...existing code...
catalog_absorbers = np.array(catalog_absorbers, dtype=float)
detected_absorbers = np.array(detected_absorbers, dtype=float)
# ...existing code...
# Create a dictionary mapping TARGETID to Z_ABS
catalog_dict = {tid: zabs for tid, zabs in zip(catalog_IDs, catalog_absorbers)}
detected_dict = {tid: zabs for tid, zabs in zip(detected_IDs, detected_absorbers)}

for i in range(len(detected_IDs)):  #loops over all detected IDs
    tid = detected_IDs[i]           #sets working ID
    if tid in catalog_dict:         #proceeds if the ID matches to catalog
        catalog_zabs = np.array(catalog_dict[tid],ndmin=1)        #Create arrays for both sets of absorbers
        detected_zabs = np.array(detected_dict[tid],ndmin=1)
        dz = np.zeros(np.size(catalog_zabs))      # Initialize dz array
        bool_match = np.zeros(np.size(catalog_zabs), dtype=bool)
        for j in range(len(detected_zabs)):     #Searching thru our detected absorbers
            current_z = detected_zabs[j]
            for k in range(np.size(catalog_zabs)):
                dz[k] = catalog_zabs[k] - current_z
                bool_match[k] = np.abs(dz[k]) < z_trh
            if sum(bool_match) > 0:             #Absorbers match, true positive
                TrueP += 1
            elif sum(bool_match) == 0:          #Absorbers do not match, false positive, we found it but they didn't
                FalseP += 1
        for j in range(len(catalog_zabs)):      #Searching thru the catalog's absorbers
            current_z = catalog_zabs[j]
            dz = detected_zabs - current_z
            bool_match = np.abs(dz) < z_trh
            if sum(bool_match) > 0:             #Absorbers match, true positive (redundant)   
                TrueP2 += 1
            elif sum(bool_match) == 0:          #Absorbers do not match, false negative, they found it but we didn't
                FalseN += 1
    else:
        detected_zabs = np.array(detected_dict[tid],ndmin=1)
        detected_zabs = detected_zabs[~np.isnan(detected_zabs)]
        FalseP += len(detected_zabs)

print("True Positive:", TrueP)
print("False Positive:", FalseP)
print("False Negative:", FalseN)
print(" ")

purity = TrueP / (TrueP + FalseP)
completeness = TrueP / (TrueP + FalseN)
print("Purity: ", purity*100, '%')
print("Completeness:", completeness*100, '%')

True Positive: 16
False Positive: 247
False Negative: 3
 
Purity:  6.083650190114068 %
Completeness: 84.21052631578947 %


In [ ]:
np.size(catalog_zabs)
range(np.size(catalog_zabs))
catalog_zabs[0]

IndexError: invalid index to scalar variable.

In [6]:
A= [1,2,3,np.nan,5,np.nan]
A = np.array(A)
print(len(A))

6
